In [3]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv(override=True)

llm = ChatGroq(model="qwen/qwen3.8-27b", temperature=0)

# ⚠️ السر هنا: نكتب chunk.content وليس chunk فقط
for chunk in llm.stream("what is the capital of egypt"):
    print(chunk.content, end="", flush=True)

The capital of Egypt is **Cairo**.

It is the largest city in Egypt and the Middle East, serving as the country's political, cultural, and economic center. Cairo is located on the Nile River and is home to many historical landmarks, including the Pyramids of Giza and the Sphinx.

In [7]:
import pandas as pd
from langchain_core.documents import Document

df = pd.read_excel("Ecommerce_Database.xlsx")

documents = []
for index, row in df.iterrows():
    # كل صف يصبح Chunk مستقل متكامل
    content = (
        f"الماركة: {row['الماركة']}\n"
        f"الموديل: {row['الموديل']}\n"
        f"المساحة: {row['المساحة (جيجابايت)']} جيجابايت\n"
        f"الرامات: {row['الرامات (جيجابايت)']} جيجابايت\n"
        f"السعر: {row['السعر (جنيه)']} جنيه"
    )
    
    # يمكن إضافة بعض الحقول كـ metadata للتصفية (Filtering) لاحقاً
    doc = Document(
        page_content=content,
        metadata={"row": index, "brand": str(row['الماركة'])}
    )
    documents.append(doc)
df.head()

,الماركة,الموديل,المساحة (جيجابايت),الرامات (جيجابايت),السعر (جنيه)
0,Apple,iPhone 15 Pro Max,256GB,8GB,68000
1,Apple,iPhone 15 Pro Max,512GB,8GB,75000
2,Apple,iPhone 15 Pro,256GB,8GB,60000
3,Apple,iPhone 15,128GB,6GB,42000
4,Apple,iPhone 14 Pro Max,256GB,6GB,58000


In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pypdf import PdfReader
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,       
    chunk_overlap=100,      
    separators=["\n\n", "\n", " ", ""] 
)



# 1. فتح ملف الـ PDF
reader = PdfReader("customerSupport.pdf")

# 2. معرفة عدد الصفحات
print("عدد الصفحات:", len(reader.pages))

# 3. استخراج النص بالكامل من جميع الصفحات
full_text = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        full_text += text + "\n"

texts = text_splitter.split_text(full_text)

print(texts[1])


عدد الصفحات: 5
:اﻟﺻوﺗﯾﺔ  ﻟﻠﻣﻛﺎﻟﻣﺔ  اﻷﺳﺎﺳﻲ  اﻟﮭﯾﻛل
ﺻﯾﺎﻧﺔ  أو  اﺧﺗﯾﺎر  ﻓﻲ  اﻟﯾوم  ﻣﺳﺎﻋدﺗك  ﯾﻣﻛﻧﻧﻲ  ﻛﯾف.  اﻵﻟﻲ  اﻟﻣﺳﺎﻋد  ﻣﻌك  ،ﻓون  أﻟﻔﺎ  ﻓﻲ  ﺑك  ًأھﻼ " (:Greeting) اﻟﺗرﺣﯾب
"؟ ھﺎﺗﻔك
"؟ ﺻﺣﯾﺢ  ،اﻵﯾﻔون  ﺑطﺎرﯾﺔ  ﻓﻲ  ﻣﺷﻛﻠﺔ  ﺗواﺟﮫ  أﻧك  ﻛﻼﻣك  ﻣن  أﻓﮭم: " ﻣﺛﺎل.  اﻟﻣﺷﻛﻠﺔ  ﻓﮭم  ﺗﺄﻛﯾد  :اﻟﻧﺷط  اﻻﺳﺗﻣﺎع
."ﻟك  اﻷﺳﻌﺎر/ اﻟﻧظﺎم  ﻣن  ﻷﺗﺣﻘق  ﻟﺣظﺔ  ﻣﻧﺣﻧﻲ  ﯾرﺟﻰ " (:Hold) اﻻﻧﺗظﺎر  ﻋﻠﻰ  اﻟﻌﻣﯾل  وﺿﻊ
ًﺷﻛرا  ؟اﻟﻣﻛﺎﻟﻣﺔ  ءإﻧﮭﺎ  ﻗﺑل  أﺧرى  ﻣﻠﺣﻘﺎت  أي  أو  ھﺎﺗﻔك  ﺑﺧﺻوص  آﺧر  اﺳﺗﻔﺳﺎر  أي  ھﻧﺎك  ھل " (:Closing) اﻟﺧﺗﺎم
."ﻻﺗﺻﺎﻟك


In [12]:
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_core.embeddings import Embeddings
from chromadb.utils import embedding_functions

# 1. إعداد دالة التضمين (Embedding) المدمجة مع ChromaDB
class ChromaDefaultEmbeddings(Embeddings):
    def __init__(self):
        self.ef = embedding_functions.DefaultEmbeddingFunction()
    def embed_documents(self, texts):
        return self.ef(texts)
    def embed_query(self, text):
        return self.ef([text])[0]

embedding_model = ChromaDefaultEmbeddings()

# 2. قراءة ملف الإكسل وتحويل كل صف إلى Document مستقل
df = pd.read_excel("Ecommerce_Database.xlsx")

documents = []
for index, row in df.iterrows():
    content = (
        f"الماركة: {row['الماركة']}\n"
        f"الموديل: {row['الموديل']}\n"
        f"المساحة: {row['المساحة (جيجابايت)']} جيجابايت\n"
        f"الرامات: {row['الرامات (جيجابايت)']} جيجابايت\n"
        f"السعر: {row['السعر (جنيه)']} جنيه"
    )
    doc = Document(
        page_content=content,
        metadata={"row_id": index, "brand": str(row['الماركة'])}
    )
    documents.append(doc)

# 3. إنشاء وحفظ الـ Vector Store في مجلد دائم (Persistent)
vector_db = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_name="ecommerce_products",
    persist_directory="./chroma_db"  # يحفظ البيانات محلياً على القرص حتى لا تضيع
)

print(f"✅ تم تخزين {len(documents)} منتج بنجاح داخل ChromaDB!")


✅ تم تخزين 30 منتج بنجاح داخل ChromaDB!


In [17]:
# 1. نطلب من الموديل تحويل السؤال لكلمات مفتاحية دقيقة للبحث
user_query = "عايز موبايل سامسونج مساحة 128"

prompt = f"""You are a helpful search assistant.
Extract the brand name, model, and storage in English keywords from the user question.
User question: "{user_query}"
Respond ONLY with the keywords (e.g. Samsung 128GB), without any explanation:"""

optimized_query = llm.invoke(prompt).content.strip()
print("🔍 البحث بالكلمات المفتاحية:", optimized_query)

# 2. البحث في ChromaDB بالكلمات المحسنة
results = vector_db.similarity_search(optimized_query, k=2)

for i, doc in enumerate(results, 1):
    print(f"\n--- النتيجة {i} ---")
    print(doc.page_content)


🔍 البحث بالكلمات المفتاحية: Samsung 128GB

--- النتيجة 1 ---
الماركة: Samsung
الموديل: Galaxy S24+
المساحة: 256GB جيجابايت
الرامات: 12GB جيجابايت
السعر: 52000 جنيه

--- النتيجة 2 ---
الماركة: Samsung
الموديل: Galaxy S24 Ultra
المساحة: 256GB جيجابايت
الرامات: 12GB جيجابايت
السعر: 65000 جنيه


In [18]:
# 1. استرجاع أفضل 5 منتجات قريبة من سؤال العميل
results = vector_db.similarity_search("Samsung 128GB", k=5)

# تجميع مواصفات الهواتف المسترجعة في سياق واحد (Context)
context = "\n---\n".join([doc.page_content for doc in results])

# 2. إرسال السياق مع سؤال العميل إلى الـ LLM ليفلتر ويجيب
system_prompt = f"""أنت موظف خدمة عملاء ودود في متجر هواتف.
أجب على سؤال العميل باللغة العربية بناءً فقط على بيانات الهواتف المتاحة أمامك.
إذا لم تجد هاتفاً مطابقاً تماماً للمواصفات المطلوبة، وضّح ذلك للعميل واقترح أقرب بديل متاح.

بيانات الهواتف المتاحة:
{context}

سؤال العميل: عايز موبايل سامسونج مساحة 128
إجابتك:"""

response = llm.invoke(system_prompt)
print(response.content)


أهلاً وسهلاً بك! سعيد جداً بمساعدتك.

بعد مراجعة المخزون المتاح لدينا حالياً، لا يوجد هاتف سامسونج بمساحة 128 جيجابايت **إلا** في موديل **Galaxy A35**، وهو متاح فعلاً بمساحة 128 جيجابايت ورامات 8 جيجابايت بسعر **18,000 جنيه**.

لكن إذا كنت تقصد هواتف الفئة العليا (مثل سلسلة S24)، فأقرب الخيارات المتاحة هي:
1. **Samsung Galaxy S24** بمساحة 256 جيجابايت ورامات 8 جيجابايت بسعر **45,000 جنيه**.
2. **Samsung Galaxy S24+** بمساحة 256 جيجابايت ورامات 12 جيجابايت بسعر **52,000 جنيه**.

هل تفضل أن نبدأ بمراجعة مواصفات **Galaxy A35** (المتاح بمساحة 128GB)، أم تفضل النظر في خيارات الفئة العليا بمساحة 256GB؟ أنا هنا لمساعدتك في الاختيار الأنسب لك! 😊


In [37]:
import pandas as pd
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_core.embeddings import Embeddings
from chromadb.utils import embedding_functions
from langchain_classic.memory import ConversationSummaryBufferMemory
import warnings
warnings.filterwarnings("ignore")

# 1. إعداد دالة الـ Embedding
class ChromaDefaultEmbeddings(Embeddings):
    def __init__(self):
        self.ef = embedding_functions.DefaultEmbeddingFunction()
    def embed_documents(self, texts):
        return self.ef(texts)
    def embed_query(self, text):
        return self.ef([text])[0]

embedding_model = ChromaDefaultEmbeddings()

# 2. تخزين مقاطع ملف الـ PDF (customerSupport.pdf) داخل ChromaDB ليكون قابلاً للبحث
pdf_docs = [Document(page_content=t, metadata={"id": i, "source": "pdf"}) for i, t in enumerate(texts)]
pdf_vector_db = Chroma.from_documents(
    documents=pdf_docs,
    embedding=embedding_model,
    collection_name="support_policies"
)

# 3. تجهيز بيانات جميع هواتف الإكسل
df = pd.read_excel("Ecommerce_Database.xlsx")
all_products_text = ""
for idx, row in df.iterrows():
    all_products_text += (
        f"- الماركة: {row['الماركة']} | الموديل: {row['الموديل']} | "
        f"المساحة: {row['المساحة (جيجابايت)']} | الرامات: {row['الرامات (جيجابايت)']} | "
        f"السعر: {row['السعر (جنيه)']} جنيه\n"
    )

# 4. إعداد الـ Memory: ConversationSummaryBufferMemory
# max_token_limit = 350: حد معقول جداً يحفظ آخر دورتين محادثة كاملة، وما زاد يلخصه تلقائياً
memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=350,
    memory_key="chat_history"
)

# 5. الدالة الشاملة للدعم الفني (الـ PDF + الإكسل + الميموري)
def ask_customer_support(user_query: str):
    print(f"\n👤 العميل: {user_query}")
    print("🤖 الدعم الفني: ", end="", flush=True)
    
    # أ. تحويل السؤال إلى كلمات بحث إنجليزية للـ Vector Store
    keyword_prompt = f"""Convert this user customer service query into 3-4 English search keywords for retrieval.
Query: "{user_query}"
Output ONLY the keywords (e.g. return policy unboxing defect), nothing else:"""
    keywords = llm.invoke(keyword_prompt).content.strip()
    
    # ب. البحث الذكي في مقاطع الـ PDF
    relevant_policy_docs = pdf_vector_db.similarity_search(keywords, k=2)
    policy_context = "\n---\n".join([d.page_content for d in relevant_policy_docs])
    
    # ج. استرجاع سياق وتاريخ المحادثة من الميموري
    chat_history = memory.load_memory_variables({})['chat_history']
    
    system_prompt = f"""أنت ممثل خدمة عملاء محترف ولطيف في متجر هواتف (ألفا فون).
أجب باللغة العربية بأسلوب راقٍ بناءً على المعلومات التالية:

[سياق وتاريخ المحادثة السابقة مع العميل]:
{chat_history}

[دليل وسياسات المتجر المستخرجة من ملف الـ PDF]:
{policy_context}

[قائمة هواتف ومخزون المتجر من ملف الإكسل]:
{all_products_text}

تعليمات صارمة:
- انتبه لتاريخ المحادثة السابقة: إذا كان العميل يسأل بالضمائر أو يتابع سؤالاً سابقاً (مثل "هو بكام"، "مساحته إيه"، "لو اشتريته")، اربط كلامه بالهاتف أو الموضوع المذكور سابقاً.
- إذا كان السؤال عن السياسات (استرجاع، ضمان، صيانة، استبدال Trade-in)، التزم بنصوص دليل وسياسات المتجر أعلاه.
- إذا كان السؤال عن شراء هاتف أو أسعار، استخرج الإجابة بدقة من قائمة المخزون.
- لا تؤلف أي معلومات من خارج النصوص المتاحة أمامك.

سؤال العميل: {user_query}
رد خدمة العملاء:"""

    full_response = ""
    for chunk in llm.stream(system_prompt):
        print(chunk.content, end="", flush=True)
        full_response += chunk.content
    print("\n")
    
    # د. حفظ السؤال والرد داخل الميموري للتحديث المستمر
    memory.save_context({"input": user_query}, {"output": full_response})


In [ ]:
# تجربة 1: العميل يسأل عن أرخص هاتف
ask_customer_support("عايز أرخص موبايل سامسونج عندكم")

# تجربة 2: سؤال متابعة بالضمير (بدون ذكر اسم الهاتف!) لاختبار الميموري
ask_customer_support("طب مساحته وراماته كام وسعره كام؟")

# تجربة 3: سؤال عن سياسة الاسترجاع (لدمج الـ PDF مع الميموري)
ask_customer_support("طب لو اشتريته وعايز استرجعه ايه الشروط؟")



👤 العميل: طب مساحته وراماته كام وسعره كام؟
🤖 الدعم الفني: أهلاً بك مجدداً في متجر ألفا فون.

بما أنك كنت تستفسر عن أرخص هاتف من سامسونج، فإن التفاصيل الخاصة بموديل **Samsung Galaxy A35** هي كالتالي:

*   **المساحة:** 128GB
*   **الرامات:** 8GB
*   **السعر:** 18,000 جنيه

هل تودّ معرفة المزيد عن مواصفات الكاميرات أو البطارية لهذا الموديل تحديداً؟



In [46]:
from voice_assistant import listen, speak

# 1. المساعد يبدأ بالترحيب الصوتي
speak("أهلاً بك في ألفا فون، كيف يمكنني مساعدتك اليوم؟")

# 2. حلقة التحدث الصوتي الكاملة
while True:
    # أ. استمع لصوتك من المايكروفون وحوله لنص
    user_voice = listen()
    
    if not user_voice:
        continue
        
    if any(w in user_voice for w in ["خروج", "مع السلامة", "باي"]):
        speak("شكراً لتواصلك مع ألفا فون، في أمان الله!")
        break
    
    # ب. إرسال النص للدالة التي بنيناها سابقاً في النوت بوك
    bot_response = ask_customer_support(user_voice)
    
    # ج. نطق الرد صوتياً عبر السماعات
    speak(bot_response)



🔊 المساعد ينطق: أهلاً بك في ألفا فون، كيف يمكنني مساعدتك اليوم؟
⚠️ خطأ أثناء تشغيل الصوت: asyncio.run() cannot be called from a running event loop

🎤 الاستماع جارٍ... تحدث الآن باللغة العربية:
⏳ جاري التعرف على الصوت...
👤 تم التقاط الصوت: عايز ارخص تليفون عندكم

👤 العميل: عايز ارخص تليفون عندكم
🤖 الدعم الفني: وعليكم السلام ورحمة الله وبركاته. أهلاً بك في متجر ألفا فون.

بناءً على قائمة المخزون المتاحة لدينا، أرخص هاتف متوفر حالياً هو **Oppo A38**.

إليك تفاصيله:
*   **المساحة:** 128GB
*   **الرامات:** 4GB
*   **السعر:** 7,500 جنيه

هذا الهاتف ينتمي للفئة الاقتصادية (Budget) وهو مناسب جداً للاستخدام اليومي الأساسي والمكالمات.

هل تود معرفة المزيد عن مواصفاته، أم تفضل مقارنته بهاتف آخر في نفس النطاق السعري؟


🎤 الاستماع جارٍ... تحدث الآن باللغة العربية:
⏳ جاري التعرف على الصوت...
❓ لم أتمكن من فهم الصوت بوضوح.

🎤 الاستماع جارٍ... تحدث الآن باللغة العربية:
⏳ جاري التعرف على الصوت...
❓ لم أتمكن من فهم الصوت بوضوح.

🎤 الاستماع جارٍ... تحدث الآن باللغة العربية:
⏳ جاري التعرف على الصوت...
👤 